# 07_03A — Walk-forward Baseline Early-Stage

## Objetivo

Validar o conjunto de features `early_stage_v1_hist` usando validação temporal com modelos baseline em `sklearn`.

Esta etapa responde:

1. As 191 features criadas nas Etapas 1 e 2 melhoram o baseline?
2. O modelo generaliza ao longo do tempo?
3. O desempenho é estável entre janelas temporais?
4. Qual modelo baseline sklearn performa melhor antes de usar bibliotecas avançadas?

## Entrada

```text
data/processed/abt_early_stage_v1_dev_hist.parquet
data/processed/abt_early_stage_v1_holdout_hist.parquet
data/processed/feature_list_early_stage_v1_hist.txt
```

## Importante

Nesta etapa ainda não usamos `category_encoders`, `feature-engine`, `CatBoost`, `OptBinning`, `skrub`, `spaCy` ou conformal prediction.

A métrica principal é `average_precision` / PR-AUC.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

pd.set_option("display.max_columns", 400)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 260)

BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "outputs" / "reports" / "modelagem_early_stage"
MODELS_DIR = BASE_DIR / "outputs" / "models" / "modelagem_early_stage"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DEV_HIST_FILE = PROCESSED_DIR / "abt_early_stage_v1_dev_hist.parquet"
HOLDOUT_HIST_FILE = PROCESSED_DIR / "abt_early_stage_v1_holdout_hist.parquet"
FEATURE_LIST_HIST_FILE = PROCESSED_DIR / "feature_list_early_stage_v1_hist.txt"

TARGET_COL = "target_banco_ganhou"
DATE_COL = "Datainicial"
RANDOM_STATE = 42

print("Setup concluído.")

## 2. Carregar bases

In [ ]:
if not DEV_HIST_FILE.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {DEV_HIST_FILE}")

if not HOLDOUT_HIST_FILE.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {HOLDOUT_HIST_FILE}")

if not FEATURE_LIST_HIST_FILE.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {FEATURE_LIST_HIST_FILE}")

df_dev = pd.read_parquet(DEV_HIST_FILE)
df_holdout = pd.read_parquet(HOLDOUT_HIST_FILE)

with open(FEATURE_LIST_HIST_FILE, "r", encoding="utf-8") as f:
    feature_list = [line.strip() for line in f if line.strip()]

df_dev[DATE_COL] = pd.to_datetime(df_dev[DATE_COL], errors="coerce")
df_holdout[DATE_COL] = pd.to_datetime(df_holdout[DATE_COL], errors="coerce")

df_dev[TARGET_COL] = df_dev[TARGET_COL].astype(int)
df_holdout[TARGET_COL] = df_holdout[TARGET_COL].astype(int)

feature_list = [c for c in feature_list if c in df_dev.columns and c in df_holdout.columns]

print("Dev:", df_dev.shape)
print("Holdout:", df_holdout.shape)
print("Features carregadas:", len(feature_list))
df_dev.head()

## 3. Funções auxiliares

In [ ]:
def save_report(df_report, filename):
    path = REPORTS_DIR / filename
    df_report.to_csv(path, index=False)
    print(f"Relatório salvo em: {path}")


def infer_feature_types(df, features):
    numeric_features = []
    categorical_features = []
    text_features = []

    for col in features:
        if col == "fe_texto_inicial_curto":
            text_features.append(col)
        elif pd.api.types.is_numeric_dtype(df[col]):
            numeric_features.append(col)
        else:
            categorical_features.append(col)

    return numeric_features, categorical_features, text_features


def make_temporal_folds(df, date_col=DATE_COL):
    folds = [
        (0.55, 0.70),
        (0.70, 0.85),
        (0.85, 1.00),
    ]

    df_sorted = df.sort_values(date_col).copy()
    dates = df_sorted[date_col]

    split_rows = []

    for i, (train_end_q, val_end_q) in enumerate(folds, start=1):
        train_end_date = dates.quantile(train_end_q)
        val_end_date = dates.quantile(val_end_q)

        train_idx = df_sorted.index[df_sorted[date_col] <= train_end_date]
        val_idx = df_sorted.index[
            (df_sorted[date_col] > train_end_date) &
            (df_sorted[date_col] <= val_end_date)
        ]

        split_rows.append({
            "fold": i,
            "train_end_q": train_end_q,
            "val_end_q": val_end_q,
            "train_start_date": df_sorted.loc[train_idx, date_col].min(),
            "train_end_date": train_end_date,
            "val_start_date": df_sorted.loc[val_idx, date_col].min() if len(val_idx) else pd.NaT,
            "val_end_date": val_end_date,
            "train_idx": train_idx,
            "val_idx": val_idx,
            "qtd_train": len(train_idx),
            "qtd_val": len(val_idx),
        })

    return split_rows


def get_threshold_metrics(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)

    return {
        "threshold": threshold,
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "pred_positive_rate": pred.mean(),
    }


def find_best_f1_threshold(y_true, proba):
    precision, recall, thresholds = precision_recall_curve(y_true, proba)

    if len(thresholds) == 0:
        return 0.5, 0, 0, 0

    precision_t = precision[:-1]
    recall_t = recall[:-1]
    f1_t = 2 * precision_t * recall_t / np.maximum(precision_t + recall_t, 1e-12)

    best_idx = np.nanargmax(f1_t)

    return thresholds[best_idx], precision_t[best_idx], recall_t[best_idx], f1_t[best_idx]


def topk_loss_risk_metrics(y_true, proba_ganho, ks=(0.01, 0.05, 0.10, 0.20)):
    score_perda = 1 - proba_ganho

    out = []
    df_tmp = pd.DataFrame({
        "y_true": y_true,
        "score_perda": score_perda
    }).sort_values("score_perda", ascending=False).reset_index(drop=True)

    total_loss = (df_tmp["y_true"] == 0).sum()
    loss_rate = (df_tmp["y_true"] == 0).mean()

    for k in ks:
        n = max(1, int(np.ceil(len(df_tmp) * k)))
        top = df_tmp.head(n)

        precision_loss_at_k = (top["y_true"] == 0).mean()
        recall_loss_at_k = (top["y_true"] == 0).sum() / total_loss if total_loss else np.nan
        lift_loss_at_k = precision_loss_at_k / loss_rate if loss_rate else np.nan

        out.append({
            "top_k": k,
            "n_top": n,
            "precision_loss_at_k": precision_loss_at_k,
            "recall_loss_at_k": recall_loss_at_k,
            "lift_loss_at_k": lift_loss_at_k,
            "base_loss_rate": loss_rate,
        })

    return pd.DataFrame(out)

## 4. Auditoria inicial das features

In [ ]:
numeric_features, categorical_features, text_features = infer_feature_types(df_dev, feature_list)

feature_type_summary = pd.DataFrame([
    {"tipo": "numeric", "qtd": len(numeric_features)},
    {"tipo": "categorical", "qtd": len(categorical_features)},
    {"tipo": "text", "qtd": len(text_features)},
])

feature_audit = pd.DataFrame({
    "feature": feature_list,
    "tipo": [
        "numeric" if f in numeric_features else
        "categorical" if f in categorical_features else
        "text"
        for f in feature_list
    ],
    "dtype_dev": [str(df_dev[f].dtype) for f in feature_list],
    "missing_rate_dev": [df_dev[f].isna().mean() for f in feature_list],
    "missing_rate_holdout": [df_holdout[f].isna().mean() for f in feature_list],
    "nunique_dev": [df_dev[f].nunique(dropna=True) for f in feature_list],
    "nunique_holdout": [df_holdout[f].nunique(dropna=True) for f in feature_list],
})

save_report(feature_audit, "18_walk_forward_feature_audit.csv")

display(feature_type_summary)
display(feature_audit.head(50))

## 5. Criar folds temporais dentro do Dev

In [ ]:
folds = make_temporal_folds(df_dev, date_col=DATE_COL)

fold_summary_rows = []

for fold in folds:
    train_idx = fold["train_idx"]
    val_idx = fold["val_idx"]

    fold_summary_rows.append({
        "fold": fold["fold"],
        "train_start_date": fold["train_start_date"],
        "train_end_date": fold["train_end_date"],
        "val_start_date": fold["val_start_date"],
        "val_end_date": fold["val_end_date"],
        "qtd_train": fold["qtd_train"],
        "qtd_val": fold["qtd_val"],
        "target_rate_train": df_dev.loc[train_idx, TARGET_COL].mean(),
        "target_rate_val": df_dev.loc[val_idx, TARGET_COL].mean(),
        "loss_rate_train": 1 - df_dev.loc[train_idx, TARGET_COL].mean(),
        "loss_rate_val": 1 - df_dev.loc[val_idx, TARGET_COL].mean(),
    })

fold_summary = pd.DataFrame(fold_summary_rows)

save_report(fold_summary, "19_walk_forward_folds_summary.csv")
fold_summary

## 6. Criar preprocessador sklearn

In [ ]:
def make_preprocessor(
    numeric_features,
    categorical_features,
    text_features,
    tfidf_max_features=1500,
    tfidf_min_df=20,
    onehot_min_frequency=20,
    scale_numeric=True,
):
    transformers = []

    if scale_numeric:
        numeric_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
    else:
        numeric_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="nao_informado")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=onehot_min_frequency
        )),
    ])

    if numeric_features:
        transformers.append(("num", numeric_pipeline, numeric_features))

    if categorical_features:
        transformers.append(("cat", categorical_pipeline, categorical_features))

    if text_features:
        text_col = text_features[0]

        text_pipeline = Pipeline(steps=[
            ("selector", FunctionTransformer(
                lambda x: x[text_col].fillna("").astype(str),
                validate=False
            )),
            ("tfidf", TfidfVectorizer(
                max_features=tfidf_max_features,
                ngram_range=(1, 2),
                min_df=tfidf_min_df
            )),
        ])

        transformers.append(("txt", text_pipeline, [text_col]))

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.3,
    )

    return preprocessor

## 7. Definir modelos baseline

In [ ]:
def make_logistic_pipeline():
    preprocessor = make_preprocessor(
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        text_features=text_features,
        tfidf_max_features=1500,
        tfidf_min_df=20,
        onehot_min_frequency=20,
        scale_numeric=True,
    )

    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])


def make_random_forest_pipeline():
    preprocessor = make_preprocessor(
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        text_features=text_features,
        tfidf_max_features=1000,
        tfidf_min_df=30,
        onehot_min_frequency=30,
        scale_numeric=False,
    )

    model = RandomForestClassifier(
        n_estimators=400,
        max_depth=18,
        min_samples_split=30,
        min_samples_leaf=15,
        max_features="sqrt",
        max_samples=0.8,
        class_weight="balanced",
        criterion="entropy",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])


def make_hist_gradient_pipeline():
    hist_numeric_features = [c for c in numeric_features if c in df_dev.columns]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
            ]), hist_numeric_features)
        ],
        remainder="drop"
    )

    model = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    )

    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])


model_factories = {
    "logistic_onehot_tfidf": make_logistic_pipeline,
    "random_forest_onehot_tfidf": make_random_forest_pipeline,
    "hist_gradient_numeric_only": make_hist_gradient_pipeline,
}

model_factories.keys()

## 8. Rodar walk-forward

In [ ]:
def evaluate_model_on_fold(model_name, pipeline, df, train_idx, val_idx, features):
    X_train = df.loc[train_idx, features].copy()
    y_train = df.loc[train_idx, TARGET_COL].copy()

    X_val = df.loc[val_idx, features].copy()
    y_val = df.loc[val_idx, TARGET_COL].copy()

    pipeline.fit(X_train, y_train)

    proba_val = pipeline.predict_proba(X_val)[:, 1]

    roc_auc = roc_auc_score(y_val, proba_val)
    pr_auc = average_precision_score(y_val, proba_val)

    best_thr, best_p, best_r, best_f1 = find_best_f1_threshold(y_val, proba_val)
    thr_05 = get_threshold_metrics(y_val, proba_val, threshold=0.5)

    result = {
        "model": model_name,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "target_rate_val": y_val.mean(),
        "loss_rate_val": 1 - y_val.mean(),
        "best_f1_threshold": best_thr,
        "best_f1_precision": best_p,
        "best_f1_recall": best_r,
        "best_f1": best_f1,
        "precision_t05": thr_05["precision"],
        "recall_t05": thr_05["recall"],
        "f1_t05": thr_05["f1"],
        "pred_positive_rate_t05": thr_05["pred_positive_rate"],
    }

    topk_loss = topk_loss_risk_metrics(y_val.values, proba_val)

    return result, topk_loss


walk_forward_results = []
topk_loss_results = []

for model_name, factory in model_factories.items():
    print("=" * 100)
    print(f"Modelo: {model_name}")

    for fold in folds:
        fold_id = fold["fold"]
        print(f"  Fold {fold_id}")

        pipeline = factory()

        result, topk_loss = evaluate_model_on_fold(
            model_name=model_name,
            pipeline=pipeline,
            df=df_dev,
            train_idx=fold["train_idx"],
            val_idx=fold["val_idx"],
            features=feature_list,
        )

        result["fold"] = fold_id
        result["qtd_train"] = fold["qtd_train"]
        result["qtd_val"] = fold["qtd_val"]
        result["train_end_date"] = fold["train_end_date"]
        result["val_start_date"] = fold["val_start_date"]
        result["val_end_date"] = fold["val_end_date"]

        walk_forward_results.append(result)

        topk_loss["model"] = model_name
        topk_loss["fold"] = fold_id
        topk_loss_results.append(topk_loss)

walk_forward_results = pd.DataFrame(walk_forward_results)
topk_loss_results = pd.concat(topk_loss_results, ignore_index=True)

save_report(walk_forward_results, "20_walk_forward_baseline_metrics.csv")
save_report(topk_loss_results, "21_walk_forward_topk_loss_risk_metrics.csv")

walk_forward_results.sort_values(["pr_auc", "roc_auc"], ascending=False)

## 9. Resumo por modelo

In [ ]:
model_summary = (
    walk_forward_results
    .groupby("model")
    .agg(
        mean_roc_auc=("roc_auc", "mean"),
        std_roc_auc=("roc_auc", "std"),
        mean_pr_auc=("pr_auc", "mean"),
        std_pr_auc=("pr_auc", "std"),
        mean_target_rate_val=("target_rate_val", "mean"),
        mean_loss_rate_val=("loss_rate_val", "mean"),
        mean_best_f1=("best_f1", "mean"),
        mean_best_f1_threshold=("best_f1_threshold", "mean"),
        mean_precision_t05=("precision_t05", "mean"),
        mean_recall_t05=("recall_t05", "mean"),
        mean_f1_t05=("f1_t05", "mean"),
    )
    .reset_index()
    .sort_values("mean_pr_auc", ascending=False)
)

save_report(model_summary, "22_walk_forward_model_summary.csv")
model_summary

## 10. Top-k de risco de perda por modelo

In [ ]:
topk_loss_summary = (
    topk_loss_results
    .groupby(["model", "top_k"])
    .agg(
        mean_precision_loss_at_k=("precision_loss_at_k", "mean"),
        std_precision_loss_at_k=("precision_loss_at_k", "std"),
        mean_recall_loss_at_k=("recall_loss_at_k", "mean"),
        mean_lift_loss_at_k=("lift_loss_at_k", "mean"),
        mean_base_loss_rate=("base_loss_rate", "mean"),
    )
    .reset_index()
    .sort_values(["top_k", "mean_precision_loss_at_k"], ascending=[True, False])
)

save_report(topk_loss_summary, "23_walk_forward_topk_loss_summary.csv")
topk_loss_summary

## 11. Selecionar melhor modelo baseline

In [ ]:
best_model_name = model_summary.iloc[0]["model"]
print("Melhor modelo por PR-AUC médio no walk-forward:", best_model_name)

## 12. Treinar melhor baseline no Dev completo e avaliar no Holdout

In [ ]:
best_pipeline = model_factories[best_model_name]()

X_dev = df_dev[feature_list].copy()
y_dev = df_dev[TARGET_COL].copy()

X_holdout = df_holdout[feature_list].copy()
y_holdout = df_holdout[TARGET_COL].copy()

best_pipeline.fit(X_dev, y_dev)

proba_holdout = best_pipeline.predict_proba(X_holdout)[:, 1]

holdout_roc_auc = roc_auc_score(y_holdout, proba_holdout)
holdout_pr_auc = average_precision_score(y_holdout, proba_holdout)

best_thr, best_p, best_r, best_f1 = find_best_f1_threshold(y_holdout, proba_holdout)
thr_05 = get_threshold_metrics(y_holdout, proba_holdout, threshold=0.5)

holdout_metrics = pd.DataFrame([{
    "model": best_model_name,
    "roc_auc": holdout_roc_auc,
    "pr_auc": holdout_pr_auc,
    "target_rate_holdout": y_holdout.mean(),
    "loss_rate_holdout": 1 - y_holdout.mean(),
    "best_f1_threshold": best_thr,
    "best_f1_precision": best_p,
    "best_f1_recall": best_r,
    "best_f1": best_f1,
    "precision_t05": thr_05["precision"],
    "recall_t05": thr_05["recall"],
    "f1_t05": thr_05["f1"],
    "pred_positive_rate_t05": thr_05["pred_positive_rate"],
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "qtd_features": len(feature_list),
}])

save_report(holdout_metrics, "24_holdout_best_baseline_metrics.csv")
holdout_metrics

## 13. Top-k no Holdout para risco de perda

In [ ]:
holdout_topk_loss = topk_loss_risk_metrics(
    y_true=y_holdout.values,
    proba_ganho=proba_holdout,
    ks=(0.01, 0.05, 0.10, 0.20)
)

holdout_topk_loss["model"] = best_model_name

save_report(holdout_topk_loss, "25_holdout_topk_loss_risk_metrics.csv")
holdout_topk_loss

## 14. Ranking do holdout por risco de perda

In [ ]:
ranking_cols = [
    "Numerodistribuicao",
    "Identificador",
    DATE_COL,
    TARGET_COL,
]

ranking_cols = [c for c in ranking_cols if c in df_holdout.columns]

ranking_holdout = df_holdout[ranking_cols].copy()
ranking_holdout["proba_banco_ganhou"] = proba_holdout
ranking_holdout["score_risco_perda"] = 1 - proba_holdout
ranking_holdout["rank_risco_perda"] = ranking_holdout["score_risco_perda"].rank(
    ascending=False,
    method="first"
).astype(int)

ranking_holdout = ranking_holdout.sort_values("score_risco_perda", ascending=False)

ranking_path = PROCESSED_DIR / "ranking_holdout_risco_perda_baseline_early_stage.parquet"
ranking_holdout.to_parquet(ranking_path, index=False)

save_report(
    ranking_holdout.head(1000),
    "26_ranking_holdout_top1000_risco_perda_baseline.csv"
)

ranking_holdout.head(20)

## 15. Classification report no Holdout

In [ ]:
pred_holdout_t05 = (proba_holdout >= 0.5).astype(int)

print("Classification report — threshold 0.5")
print(classification_report(y_holdout, pred_holdout_t05))

print("Confusion matrix — threshold 0.5")
print(confusion_matrix(y_holdout, pred_holdout_t05))

## 16. Salvar resumo final da Etapa 3A

In [ ]:
summary_step_3a = pd.DataFrame([{
    "best_model_walk_forward": best_model_name,
    "best_model_mean_pr_auc_wf": model_summary.loc[model_summary["model"] == best_model_name, "mean_pr_auc"].iloc[0],
    "best_model_mean_roc_auc_wf": model_summary.loc[model_summary["model"] == best_model_name, "mean_roc_auc"].iloc[0],
    "holdout_pr_auc": holdout_pr_auc,
    "holdout_roc_auc": holdout_roc_auc,
    "target_rate_holdout": y_holdout.mean(),
    "loss_rate_holdout": 1 - y_holdout.mean(),
    "qtd_features": len(feature_list),
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "ranking_holdout_path": str(ranking_path),
}])

save_report(summary_step_3a, "27_summary_step_3a_walk_forward_baseline.csv")
summary_step_3a.T

# O que verificar após rodar

Traga para a próxima iteração:

1. `fold_summary`
2. `model_summary`
3. `holdout_metrics`
4. `holdout_topk_loss`
5. `summary_step_3a.T`
6. Se houve lentidão em algum modelo.

## Próxima etapa

Se a Etapa 3A estiver ok, seguimos para:

```text
Etapa 3B — Encoders avançados sem leakage
```

Nela entram:

- `category_encoders.CatBoostEncoder`;
- `category_encoders.JamesSteinEncoder`;
- `category_encoders.MEstimateEncoder`;
- `feature-engine.RareLabelEncoder`;
- opcionalmente `feature-engine.WoEEncoder`.